In [17]:
import pandas as pd

# Load datasets
df = pd.read_csv("processed_dataset.csv")
school_df = pd.read_csv("school_zone.csv")

# Clean CASE_ID
df['CASE_ID'] = df['CASE_ID'].astype(str).str.strip().str.replace(".0", "", regex=False)
school_df['CASE_ID'] = school_df['CASE_ID'].astype(str).str.strip().str.replace(".0", "", regex=False)

# Create set of school CASE_IDs
school_ids = set(school_df['CASE_ID'].drop_duplicates())

# Create indicator column
df['school_zone'] = df['CASE_ID'].isin(school_ids).astype(int)

# 🔍 Check before overwrite
print(df['school_zone'].value_counts())

df['label'] = (df['SEVERITY'] >= 3).astype(int)
df.drop(columns ='SEVERITY')

# 💾 OVERWRITE original file
df.to_csv("processed_dataset.csv", index=False)

school_zone
0    6349
1    2954
Name: count, dtype: int64


Protocal A cross city OOD 

### Creating splits for Cross City OOD

In [18]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("processed_dataset.csv")

# --- Optional: convert severity to binary if needed ---
# Uncomment if your label is not already binary
# df['label'] = (df['SEVERITY'] >= 3).astype(int)

label_col = 'label'  # or 'label' if you created one
city_col = 'CITY'

# --- Create splits ---
def create_cross_city_splits(df, city_col, label_col, seed=0):
    cities = df[city_col].unique()
    splits = []

    for test_city in cities:
        # --- Test set (one city) ---
        test_idx = df[df[city_col] == test_city].index

        # --- Train + Val ---
        train_val_df = df[df[city_col] != test_city]

        # --- Stratified split ---
        train_idx, val_idx = train_test_split(
            train_val_df.index,
            test_size=0.15,
            stratify=train_val_df[label_col],
            random_state=seed
        )

        splits.append({
            "test_city": str(test_city),
            "train_idx": list(map(int, train_idx)),
            "val_idx": list(map(int, val_idx)),
            "test_idx": list(map(int, test_idx))
        })

    return splits

# --- Generate splits for 5 seeds ---
all_splits = {}

for seed in range(5):
    all_splits[seed] = create_cross_city_splits(df, city_col, label_col, seed)

# --- Save splits ---
os.makedirs("splits", exist_ok=True)

for seed in all_splits:
    for i, split in enumerate(all_splits[seed]):
        filename = f"splits/city_seed{seed}_split{i}.json"
        with open(filename, "w") as f:
            json.dump(split, f)

print("✅ Splits saved successfully!")

✅ Splits saved successfully!


### Creating Temporal OOD Splits

In [19]:
# --- Column name ---
year_col = "ACCIDENT_YEAR"  # based on your dataset

# --- Create split ---
train_idx = df[(df[year_col] >= 2013) & (df[year_col] <= 2019)].index
val_idx   = df[(df[year_col] >= 2020) & (df[year_col] <= 2021)].index
test_idx  = df[(df[year_col] >= 2022) & (df[year_col] <= 2024)].index

split = {
    "train_idx": list(map(int, train_idx)),
    "val_idx": list(map(int, val_idx)),
    "test_idx": list(map(int, test_idx))
}

# --- Save ---
os.makedirs("splits", exist_ok=True)

with open("splits/temporal_split.json", "w") as f:
    json.dump(split, f)

print("✅ Temporal split saved!")

✅ Temporal split saved!


In [20]:
print("Train years:", sorted(df.loc[train_idx]['ACCIDENT_YEAR'].unique()))
print("Val years:", sorted(df.loc[val_idx]['ACCIDENT_YEAR'].unique()))
print("Test years:", sorted(df.loc[test_idx]['ACCIDENT_YEAR'].unique()))

print("\nSizes:")
print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

Train years: [2014, 2015, 2016, 2017, 2018, 2019]
Val years: [2020, 2021]
Test years: [2022, 2023, 2024]

Sizes:
Train: 5173
Val: 1465
Test: 2665


### Creating Policy Shift (School Zones) Splits 

In [21]:
from sklearn.model_selection import train_test_split

# --- Split based on school_zone ---
train_val_df = df[df['school_zone'] == 1]
test_df      = df[df['school_zone'] == 0]

# --- Stratified train/val split ---
train_idx, val_idx = train_test_split(
    train_val_df.index,
    test_size=0.15,
    stratify=train_val_df['label'],
    random_state=0
)

test_idx = test_df.index

# --- Save split ---
split = {
    "train_idx": list(map(int, train_idx)),
    "val_idx": list(map(int, val_idx)),
    "test_idx": list(map(int, test_idx))
}

os.makedirs("splits", exist_ok=True)

with open("splits/policy_split.json", "w") as f:
    json.dump(split, f)

print("✅ Policy split saved!")

✅ Policy split saved!


### Graph Construction

In [22]:
import scipy.sparse as sp
from sklearn.neighbors import NearestNeighbors
import pickle

# =========================
# Load dataset
# =========================
df = pd.read_csv("processed_dataset.csv")

# =========================
# Extract coordinates
# =========================
coords = df[['LATITUDE', 'LONGITUDE']].values
n = len(coords)

# =========================
# Build kNN graph
# =========================
k = 20

nbrs = NearestNeighbors(n_neighbors=k + 1, algorithm='ball_tree')
nbrs.fit(coords)

distances, indices = nbrs.kneighbors(coords)

# Remove self-loops (first neighbor is itself)
distances = distances[:, 1:]
indices = indices[:, 1:]

# =========================
# Compute Gaussian weights
# =========================
# σ = average neighbor distance (robust default)
sigma = np.mean(distances)

weights = np.exp(-(distances ** 2) / (sigma ** 2))

# =========================
# Build sparse adjacency matrix
# =========================
rows = np.repeat(np.arange(n), k)
cols = indices.flatten()
vals = weights.flatten()

A = sp.coo_matrix((vals, (rows, cols)), shape=(n, n))

# =========================
# Symmetrize graph (undirected)
# =========================
A = (A + A.T) / 2

# =========================
# Compute Laplacian
# =========================
D = sp.diags(A.sum(axis=1).A1)
L = D - A

# =========================
# Save graph
# =========================
graph_data = {
    "adjacency": A.tocsr(),   # efficient format
    "laplacian": L.tocsr(),
    "k": k,
    "sigma": float(sigma)
}

with open("graph.pkl", "wb") as f:
    pickle.dump(graph_data, f)

print("✅ Graph constructed and saved as graph.pkl")

✅ Graph constructed and saved as graph.pkl


In [23]:
print("Nodes:", n)
print("Edges (non-zero entries):", A.nnz)
print("k:", k)
print("sigma:", sigma)

# Check symmetry
print("Symmetry check (should be ~0):", (A - A.T).nnz)

# Weight range
print("Min weight:", weights.min())
print("Max weight:", weights.max())

Nodes: 9303
Edges (non-zero entries): 229100
k: 20
sigma: 0.014425909486433033
Symmetry check (should be ~0): 0
Min weight: 0.0
Max weight: 1.0


### Non-Graph Baseline Models 

In [2]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'  # important for imbalance
)

In [3]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)  # binary
        )

    def forward(self, x):
        return self.net(x)

In [5]:
pip install xgboost

  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
  Using cached nvidia_nccl_cu12-2.29.7-py3-none-manylinux_2_18_x86_64.whl (293.6 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:

import xgboost as xgb

# will be updated for each training regime 
imbalance_ratio = None 

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio
)

In [7]:
df = pd.read_csv("processed_dataset.csv")

# Clean CITY column
city_clean = df['CITY'].astype(str).str.strip().str.upper()

count = (city_clean == "UNINCORPORATED").sum()

print("Count of UNINCORPORATED:", count)

NameError: name 'pd' is not defined

## Training Baselines in Cross City OOD 

#### Logistic Regression Baseline Training 

In [2]:
import json
import glob
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

# -------------------------
# Choose seed
# -------------------------
seed = 0  # change as needed

# -------------------------
# Load data
# -------------------------
df = pd.read_csv("processed_dataset.csv")

label_col = "label"
city_col = "CITY"

X_all = df.drop(columns=[label_col, city_col])
X_all = X_all.select_dtypes(include=[np.number])
y_all = df[label_col].values

# -------------------------
# Load splits for this seed
# -------------------------
split_files = sorted(glob.glob(f"splits/city_seed{seed}_split*.json"))

results = []

for split_file in split_files:
    with open(split_file, "r") as f:
        split = json.load(f)

    train_idx = split["train_idx"]
    test_idx = split["test_idx"]
    test_city = split["test_city"]

    # --- Data ---
    X_train, y_train = X_all.iloc[train_idx], y_all[train_idx]
    X_test, y_test = X_all.iloc[test_idx], y_all[test_idx]

    # --- Model (no scaling since data already preprocessed) ---
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # --- Evaluate ---
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred, average="weighted")
    acc = accuracy_score(y_test, y_pred)

    results.append({
        "city": test_city,
        "f1": f1,
        "acc": acc
    })

    print(f"{test_city} → F1: {f1:.4f}, Acc: {acc:.4f}")

# -------------------------
# Aggregate (Protocol A metrics)
# -------------------------
results_df = pd.DataFrame(results)

print("\n===== Cross-City OOD Results =====")
print(f"MeanEnvF1:   {results_df['f1'].mean():.4f}")
print(f"WorstEnvF1:  {results_df['f1'].min():.4f}")
print(f"MeanEnvAcc:  {results_df['acc'].mean():.4f}")
print(f"WorstEnvAcc: {results_df['acc'].min():.4f}")

UNINCORPORATED → F1: 0.2699, Acc: 0.4410
SAN BERNARDINO → F1: 0.0915, Acc: 0.2380
RANCHO CUCAMONGA → F1: 0.0472, Acc: 0.1659
CHINO → F1: 0.0312, Acc: 0.1329
MONTCLAIR → F1: 0.0421, Acc: 0.1561
YUCCA VALLEY → F1: 0.2209, Acc: 0.3922
HESPERIA → F1: 0.1341, Acc: 0.2947
UPLAND → F1: 0.0334, Acc: 0.1378
ADELANTO → F1: 0.1470, Acc: 0.3103
BARSTOW → F1: 0.2695, Acc: 0.4406
BIG BEAR LAKE → F1: 0.0725, Acc: 0.2093
APPLE VALLEY → F1: 0.2190, Acc: 0.3901
COLTON → F1: 0.1153, Acc: 0.2707
CHINO HILLS → F1: 0.0404, Acc: 0.1527
TWENTYNINE PALMS → F1: 0.3903, Acc: 0.5500
LOMA LINDA → F1: 0.1402, Acc: 0.3021
GRAND TERRACE → F1: 0.0278, Acc: 0.1250
NEEDLES → F1: 0.1000, Acc: 0.2500
YUCAIPA → F1: 0.1160, Acc: 0.2715
ONTARIO → F1: 0.0752, Acc: 0.2136
RIALTO → F1: 0.1055, Acc: 0.2575
REDLANDS → F1: 0.0490, Acc: 0.1692
VICTORVILLE → F1: 0.1457, Acc: 0.3088
HIGHLAND → F1: 0.1259, Acc: 0.2844
FONTANA → F1: 0.0670, Acc: 0.2005

===== Cross-City OOD Results =====
MeanEnvF1:   0.1231
WorstEnvF1:  0.0278
MeanEnvA

In [1]:
results_df

NameError: name 'results_df' is not defined

#### Training Baselines in Temporal OOD 

#### Training Baselines in Policy Shift 